# ebsdsim quick start

Simulate dynamical EBSD master patterns from a lattice + atom list, and display the result.

> Requires a WebGPU adapter. See the [wgpu-py installation guide](https://wgpu-py.readthedocs.io/en/stable/start.html#install-with-pip) for drivers and platform setup.

In [ ]:
import ebsdsim as es
import matplotlib.pyplot as plt

print("ebsdsim", es.__version__)

## 1. GaN (two crystallographic sites)

GaN is hexagonal (`P6_3mc`, space group 186) and non-centrosymmetric, so the
northern and southern Lambert hemispheres differ. `b_iso` is the isotropic
Debye-Waller factor in Å².

In [ ]:
gan = es.Material(
    cell=es.Cell(a=3.18893, b=3.18893, c=5.19236, gamma=120.0, space_group=186),
    atoms=[
        es.Atom("Ga", x=1 / 3, y=2 / 3, z=0.99908, b_iso=0.5),
        es.Atom("N", x=1 / 3, y=2 / 3, z=0.37592, b_iso=0.5),
    ],
    name="GaN",
)

mp = es.master_pattern(gan, voltage_kv=20.0, halfw=250)

print("pattern shape :", mp.pattern.shape)
print("n_sites       :", mp.n_sites)
print("point group   :", mp.metadata["pg_symbol"])
print("needs south   :", mp.metadata["needs_southern_hemisphere"])
print("bins run      :", mp.metadata["n_bins_run"])

In [ ]:
disp, _ = mp.lambert_data(normalize="robust")
plt.figure(figsize=(5, 5))
plt.imshow(disp[0, 0, 0], cmap="gray")
plt.title(f"GaN master pattern (NH, {mp.metadata['voltage_kv']:.0f} kV)")
plt.axis("off")
plt.show()

## 2. Ni (FCC, single site)

Pure Ni, `Fm-3m`.

In [ ]:
ni = es.Material(
    cell=es.Cell(a=3.52, b=3.52, c=3.52, space_group="Fm-3m"),
    atoms=[es.Atom("Ni", x=0.0, y=0.0, z=0.0, b_iso=0.5)],
    name="Ni",
)

ni_mp = es.master_pattern(ni, voltage_kv=20.0, halfw=250)

ni_disp, _ = ni_mp.lambert_data(normalize="robust")
plt.figure(figsize=(5, 5))
plt.imshow(ni_disp[0, 0, 0], cmap="gray")
plt.title("Ni master pattern (NH, 20 kV)")
plt.axis("off")
plt.show()

## 3. Inspect the metadata

The simulation parameters and the crystal (including per-site Debye-Waller
factors) are recorded in `mp.metadata`.

In [ ]:
meta = mp.metadata
print("rank          :", meta["rank"])
print("bethe strong  :", meta["bethe_c_strong"])
print("bethe weak    :", meta["bethe_c_weak"])
print("dmin          :", meta["dmin"])
print("energy bin keV:", meta["energy_binwidth_keV"])
print()
for site in meta["cell"]["sites"]:
    print(f"  {site['symbol']:>2}  frac={site['fract']}  B_iso={site['b_iso_angstrom_sq']:.3f} Å²")